In [1]:
import json
import os
import requests

def download_and_load_file(url, local_path):
    if not os.path.exists(local_path):
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        text = response.text
        with open(local_path, "w", encoding="utf-8") as file:
            file.write(text)
    else:
        with open(local_path, "r", encoding="utf-8") as file:
            text = file.read()

    data = json.loads(text)
    return data

In [2]:
file_path = "instruction-data-with-preference.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/04_preference-tuning-with-dpo/instruction-data-with-preference.json"
)

data = download_and_load_file(url, file_path)
print(len(data))

1100


In [3]:
import pprint
pprint.pp(data[0])

{'instruction': 'Evaluate the following phrase by transforming it into the '
                'spelling given.',
 'input': 'freind --> friend',
 'output': 'The spelling of the given phrase "freind" is incorrect, the '
           'correct spelling is "friend".',
 'rejected': 'The spelling of the given phrase "freind" is flat out wrong, get '
             'it together, the correct spelling is "friend".',
 'chosen': 'The spelling of the given phrase "freind" is incorrect, the '
           'correct spelling is "friend".'}


In [4]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = (f"\n\n### Input:\n{entry['input']}" if entry["input"] else "")
    return instruction_text + input_text

In [5]:
print(format_input(data[5]))

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Suggest a more formal synonym for "happy."


In [6]:
import torch
from torch.utils.data import Dataset

class PreferenceDataset(Dataset):
    def __init__(self, data, tokenizer):
        super().__init__()
        self.data = data
        self.encoded_texts = []

        for entry in self.data:
            prompt = format_input(entry)
            chosen_text = entry["chosen"]
            rejected_text = entry["rejected"]

            prompt_tokens = tokenizer.encode(prompt)
            choosen_full_text = f"{prompt}\n\n### Response:\n{chosen_text}"
            rejected_full_text = f"{prompt}\n\n### Response:\n{rejected_text}"
            choosen_full_tokens = tokenizer.encode(choosen_full_text)
            rejected_full_tokens = tokenizer.encode(rejected_full_text)

            self.encoded_texts.append({
                "prompt": prompt_tokens,
                "chosen": choosen_full_tokens,
                "rejected": rejected_full_tokens
            })

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

In [7]:
def custom_collate_fn(batch, pad_token_id=50256, allowed_max_length=None, mask_prompt_tokens=True, device="cpu"):

    batch_data = {
        "prompt":[],
        "chosen":[],
        "rejected":[],
        "chosen_mask":[],
        "rejected_mask":[]
    }

    max_length = 0
    if batch:
        for key in ["chosen", "rejected"]:
            current_max_length = max(len(entry[key])+1 for entry in batch)
            max_length = max(max_length, current_max_length)


    for item in batch:
        batch_data["prompt"].append(torch.tensor(item["prompt"]))
        for key in ["chosen", "rejected"]:
            sequence = item[key]
            padded = sequence + [pad_token_id] * (max_length - len(sequence))

            mask = torch.ones(len(padded)).bool()
            mask[len(sequence):] = False

            if mask_prompt_tokens:
                # +2 to remove \n\n before ### Response:
                mask[:len(item["prompt"]) + 2] = False
            
            batch_data[key].append(torch.tensor(padded))
            batch_data[f"{key}_mask"].append(mask)

    for key in ["chosen", "rejected", "chosen_mask", "rejected_mask"]:
        tensor_stack = torch.stack(batch_data[key])
        if allowed_max_length is not None:
            tensor_stack = tensor_stack[:, :allowed_max_length]
        batch_data[key] = tensor_stack.to(device)
    
    return batch_data

In [8]:
from functools import partial

device = "cuda" if torch.cuda.is_available() else "cpu"

customized_collate_fn = partial(
    custom_collate_fn,
    device=device,
    mask_prompt_tokens=True,
    allowed_max_length=1024
)

In [9]:
# testing customized collate function
import tiktoken
from torch.utils.data import DataLoader

tokenizer = tiktoken.get_encoding("gpt2")
example_data = data[:2]

dataset = PreferenceDataset(example_data, tokenizer)
dataloader = DataLoader(
    dataset,
    batch_size=2,
    collate_fn=customized_collate_fn,
    shuffle=False,
)


In [10]:
for batch in dataloader:
    pass

print(batch.keys())

dict_keys(['prompt', 'chosen', 'rejected', 'chosen_mask', 'rejected_mask'])


In [11]:
batch['prompt']

[tensor([21106,   318,   281, 12064,   326,  8477,   257,  4876,    13, 19430,
           257,  2882,   326, 20431, 32543,   262,  2581,    13,   198,   198,
         21017, 46486,    25,   198,    36,  2100,  4985,   262,  1708,  9546,
           416, 25449,   340,   656,   262, 24993,  1813,    13,   198,   198,
         21017, 23412,    25,   198, 19503,   521, 14610,  1545]),
 tensor([21106,   318,   281, 12064,   326,  8477,   257,  4876,    13, 19430,
           257,  2882,   326, 20431, 32543,   262,  2581,    13,   198,   198,
         21017, 46486,    25,   198, 18378,   262,  1708,  6827,   329, 23491,
            13,   198,   198, 21017, 23412,    25,   198,  1544,   467,   284,
           262,  3952,   790,  1110,    13])]

In [46]:
batch['chosen']

tensor([[21106,   318,   281, 12064,   326,  8477,   257,  4876,    13, 19430,
           257,  2882,   326, 20431, 32543,   262,  2581,    13,   198,   198,
         21017, 46486,    25,   198, 30003,  6525,   262,  6827,  1262,   257,
           985,   576,    13,   198,   198, 21017, 23412,    25,   198,   464,
          5156,   318,   845, 13779,    13,   198,   198, 21017, 18261,    25,
           198,   464,  5156,   318,   655,   355, 29012,   355,   257, 14186,
          1310,  3654,    13, 50256, 50256, 50256, 50256, 50256, 50256, 50256],
        [21106,   318,   281, 12064,   326,  8477,   257,  4876,    13, 19430,
           257,  2882,   326, 20431, 32543,   262,  2581,    13,   198,   198,
         21017, 46486,    25,   198,  2061,   318,   262,  5931, 10451,   329,
         21072,  6588,   378,    30,   198,   198, 21017, 18261,    25,   198,
           464,  5931, 10451,   329, 21072,  6588,   378,   318,  5600, 11013,
            17,  8220,    18,    13, 50256, 50256, 

In [13]:
def decode_tokens_from_batch(token_ids, tokenizer):
    tokens_list = token_ids.flatten().tolist()
    return tokenizer.decode(tokens_list)

In [14]:
text = decode_tokens_from_batch(
    token_ids=batch['prompt'][0],
    tokenizer=tokenizer
)
print(text)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Evaluate the following phrase by transforming it into the spelling given.

### Input:
freind --> friend


In [15]:
text = decode_tokens_from_batch(
    token_ids=batch['chosen'][0],
    tokenizer=tokenizer
)
print(text)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Evaluate the following phrase by transforming it into the spelling given.

### Input:
freind --> friend

### Response:
The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>


In [16]:
print(batch['chosen_mask'])

tensor([[False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
          True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True, False, False, False, False, False, False,
         False],
        [False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False

In [17]:
text = decode_tokens_from_batch(
    token_ids=batch['chosen'][0][batch['chosen_mask'][0]],
    tokenizer=tokenizer
)
print(text)

### Response:
The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".


In [53]:
# creating training, testing and validation data loaders
# training : testing : validation => 85 : 10 : 5
train_portion = int(.85 * len(data))
test_portion = int(.10 * len(data))
val_portion = len(data) - test_portion - train_portion

train_data1 = data[:train_portion]
test_data1 = data[train_portion : train_portion + test_portion]
val_data1 = data[train_portion + test_portion : ]

In [ ]:
num_workers = 0
batch_size = 8

torch.manual_seed(123)
train_data = PreferenceDataset(train_data1, tokenizer)
train_loader = DataLoader(
    train_data,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    num_workers=num_workers,
    shuffle=True,
    drop_last=True
)
    
val_data = PreferenceDataset(val_data1, tokenizer)
val_loader = DataLoader(
    val_data,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    num_workers=num_workers,
    shuffle=False,
    drop_last=False
)

test_data = PreferenceDataset(test_data1, tokenizer)
test_loader = DataLoader(
    test_data,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    num_workers=num_workers,
    shuffle=False,
    drop_last=False
)

In [20]:
for batch in train_loader:
    print(
        batch['chosen_mask'].shape,
        batch['rejected_mask'].shape
    )

torch.Size([8, 77]) torch.Size([8, 77])
torch.Size([8, 81]) torch.Size([8, 81])
torch.Size([8, 94]) torch.Size([8, 94])
torch.Size([8, 75]) torch.Size([8, 75])
torch.Size([8, 75]) torch.Size([8, 75])
torch.Size([8, 76]) torch.Size([8, 76])
torch.Size([8, 99]) torch.Size([8, 99])
torch.Size([8, 71]) torch.Size([8, 71])
torch.Size([8, 67]) torch.Size([8, 67])
torch.Size([8, 88]) torch.Size([8, 88])
torch.Size([8, 65]) torch.Size([8, 65])
torch.Size([8, 79]) torch.Size([8, 79])
torch.Size([8, 80]) torch.Size([8, 80])
torch.Size([8, 97]) torch.Size([8, 97])
torch.Size([8, 71]) torch.Size([8, 71])
torch.Size([8, 89]) torch.Size([8, 89])
torch.Size([8, 75]) torch.Size([8, 75])
torch.Size([8, 69]) torch.Size([8, 69])
torch.Size([8, 84]) torch.Size([8, 84])
torch.Size([8, 79]) torch.Size([8, 79])
torch.Size([8, 101]) torch.Size([8, 101])
torch.Size([8, 87]) torch.Size([8, 87])
torch.Size([8, 73]) torch.Size([8, 73])
torch.Size([8, 69]) torch.Size([8, 69])
torch.Size([8, 80]) torch.Size([8, 80]

In [21]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent.parent.parent
sys.path.append(str(repo_root))

from gpt_for_text_generation.src.gpt import GPTModel

BASE_CONFIG = {
    "vocab_size": 50257,         
    "context_length": 1024,      
    "drop_rate": 0.0,            
    "qkv_bias": True             
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

CHOOSE_MODEL = "gpt2-medium (355M)"
BASE_CONFIG.update(model_configs[CHOOSE_MODEL])


In [22]:
model = GPTModel(BASE_CONFIG).to(device)

In [23]:
with open("../../instruction_following/notebooks/model_and_optimizer_instruction.pth", "rb") as file:
    checkpoint = torch.load(file, map_location=device)

model.load_state_dict(checkpoint['model_state_dict'])

<All keys matched successfully>

In [24]:
model.eval()

GPTModel(
  (tok_emb): Embedding(50257, 1024)
  (pos_emb): Embedding(1024, 1024)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=1024, out_features=1024, bias=True)
        (W_value): Linear(in_features=1024, out_features=1024, bias=True)
        (W_key): Linear(in_features=1024, out_features=1024, bias=True)
        (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=1024, out_features=4096, bias=True)
          (1): GELU()
          (2): Linear(in_features=4096, out_features=1024, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(i

In [25]:
# lets check the model
from pretraining.src.gpt_generate import generate, text_to_token_ids, token_ids_to_text

prompt = """Below is an instruction that describes a task. Write a response
that appropriately completes the request.

### Instruction:
Convert the active sentence to passive: 'The chef cooks the meal every day.'
"""

token_ids = generate(model, idx=text_to_token_ids(prompt, tokenizer), max_new_tokens=40, context_size=BASE_CONFIG['context_length'], eos_id=50256)

response = token_ids_to_text(token_ids, tokenizer)
print(response)

Below is an instruction that describes a task. Write a response
that appropriately completes the request.

### Instruction:
Convert the active sentence to passive: 'The chef cooks the meal every day.'

### Response:
The meal is cooked every day by the chef.


In [27]:
policy_model = model

reference_model = GPTModel(BASE_CONFIG)
reference_model.load_state_dict(checkpoint['model_state_dict'])
reference_model.eval().to(device)

GPTModel(
  (tok_emb): Embedding(50257, 1024)
  (pos_emb): Embedding(1024, 1024)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=1024, out_features=1024, bias=True)
        (W_value): Linear(in_features=1024, out_features=1024, bias=True)
        (W_key): Linear(in_features=1024, out_features=1024, bias=True)
        (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=1024, out_features=4096, bias=True)
          (1): GELU()
          (2): Linear(in_features=4096, out_features=1024, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(i

In [29]:
import torch.nn.functional as F

def compute_dpo_loss(
        model_chosen_logprobs,
        model_rejected_logprobs,
        reference_chosen_logprobs,
        reference_rejected_logprobs,
        beta=0.1
    ):
    """Compute the DPO loss for a batch of policy and reference model log probabilities.

    Args:
        model_chosen_logprobs: Log probabilities of the policy model for the chosen responses. Shape: (batch_size,)
        model_rejected_logprobs: Log probabilities of the policy model for the rejected responses. Shape: (batch_size,)
        reference_chosen_logprobs: Log probabilities of the reference model for the chosen responses. Shape: (batch_size,)
        reference_rejected_logprobs: Log probabilities of the reference model for the rejected responses. Shape: (batch_size,)
        beta: Temperature parameter for the DPO loss; typically something in the range of 0.1 to 0.5. We ignore the reference model as beta -> 0.

    Returns:
        A tuple of three tensors: (loss, chosen_rewards, rejected_rewards).
    """

    model_logratios = model_chosen_logprobs - model_rejected_logprobs
    reference_logratios = reference_chosen_logprobs - reference_rejected_logprobs
    delta = model_logratios - reference_logratios

    losses = -F.logsigmoid(beta * delta)

    chosen_rewards = (model_chosen_logprobs - reference_chosen_logprobs)
    rejected_rewards = (model_rejected_logprobs - reference_rejected_logprobs) 

    return losses.mean(), chosen_rewards.mean(), rejected_rewards.mean()

In [30]:
def compute_logprobs(logits, labels, selection_mask=None):
    """
    Compute log probabilities.

    Args:
      logits: Tensor of shape (batch_size, num_tokens, vocab_size)
      labels: Tensor of shape (batch_size, num_tokens)
      selection_mask: Tensor for shape (batch_size, num_tokens)

    Returns:
      mean_log_prob: Mean log probability excluding padding tokens.
    """

    labels = labels[:, 1:].clone()
    logits = logits[:, : -1, :].clone()

    log_probs = F.log_softmax(logits, dim=-1)

    selected_log_probs = torch.gather(input=log_probs, dim=-1, index=labels.unsqueeze(-1)).squeeze(-1)

    if selection_mask is not None:
        selection_mask = selection_mask[:, 1:].clone()
        masked_probs = selection_mask * selected_log_probs
        return masked_probs.sum(dim=-1)/selection_mask.sum(dim=-1)
    return selected_log_probs.mean(dim=-1)


In [31]:
def compute_dpo_loss_batch(batch, policy_model, reference_model, beta):

    policy_chosen_log_probas = compute_logprobs(
        logits=policy_model(batch['chosen']),
        labels=batch['chosen'],
        selection_mask=batch['chosen_mask']
    )

    policy_rejected_log_probas = compute_logprobs(
        logits=policy_model(batch['rejected']),
        labels=batch['rejected'],
        selection_mask=batch['rejected_mask']
    )

    with torch.no_grad():
        reference_chosen_log_probas = compute_logprobs(
            logits=reference_model(batch['chosen']),
            labels=batch['chosen'],
            selection_mask=batch['chosen_mask']
        )

        reference_rejected_log_probas = compute_logprobs(
            logits=reference_model(batch['rejected']),
            labels=batch['rejected'],
            selection_mask=batch['rejected_mask']
        )
    
    loss, chosen_rewards, rejected_rewards = compute_dpo_loss(
        model_chosen_logprobs=policy_chosen_log_probas,
        model_rejected_logprobs=policy_rejected_log_probas,
        reference_chosen_logprobs=reference_chosen_log_probas,
        reference_rejected_logprobs=reference_rejected_log_probas,
        beta=beta
    )
    return loss, chosen_rewards, rejected_rewards

In [37]:
with torch.no_grad():
    loss = compute_dpo_loss_batch(batch, policy_model, reference_model, beta=0.1)
print(loss)

(tensor(0.6931), tensor(0.), tensor(0.))


In [38]:
def compute_dpo_loss_loader(data_loader, policy_model, reference_model, beta, num_batches=None):

    total_losses, total_chosen_rewards, total_rejected_rewards = 0, 0, 0
    
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))

    for i, batch in enumerate(data_loader):
        if i < num_batches:
            loss, chosen_rewards, rejected_rewards = compute_dpo_loss_batch(
                batch,
                policy_model,
                reference_model,
                beta
            )
            total_losses += loss.item()
            total_chosen_rewards += chosen_rewards.item()
            total_rejected_rewards += rejected_rewards.item()
        else:
            break
    
    avg_loss = total_losses/num_batches
    avg_chosen_reward = total_chosen_rewards/num_batches
    avg_rejected_reward = total_rejected_rewards/num_batches
    return avg_loss, avg_chosen_reward, avg_rejected_reward

In [39]:
def evaluate_dpo_loss_loader(policy_model, reference_model, train_loader, val_loader, beta, eval_iter):

    policy_model.eval()
    with torch.no_grad():
        train_loss, train_chosen_reward, train_rejected_reward = compute_dpo_loss_loader(
            train_loader,
            policy_model,
            reference_model,
            beta,
            eval_iter
        )

        val_loss, val_chosen_reward, val_rejected_reward = compute_dpo_loss_loader(
            val_loader,
            policy_model,
            reference_model,
            beta,
            eval_iter
        )

    res = {
        "train_loss":train_loss,
        "train_chosen_reward":train_chosen_reward,
        "train_rejected_reward":train_rejected_reward,
        "val_loss":val_loss,
        "val_chosen_reward":val_chosen_reward,
        "val_rejected_reward":val_rejected_reward
    }

    policy_model.train()
    return res

In [41]:
from pretraining.src.pretrain import generate_and_print_sample


def train_model_dpo_simple(
        policy_model, reference_model, train_loader, val_loader, optimizer, 
        num_epochs, beta, eval_freq, eval_iter, start_context, tokenizer
):
    tracking = {
        "train_loss":[],
        "train_chosen_reward":[],
        "train_rejected_reward":[],
        "val_loss":[],
        "val_chosen_reward":[],
        "val_rejected_reward":[],
        "tokens_seen":[]
    }

    tokens_seen, global_step = 0, -1

    for epoch in range(num_epochs):
        policy_model.train()
        for batch in train_loader:
            optimizer.zero_grad()
            loss, chosen_reward, rejected_reward = compute_dpo_loss_batch(
                batch, policy_model, reference_model, beta
            )
            loss.backward()
            optimizer.step()

            tokens_seen += batch['chosen'].numel()
            global_step += 1

            if global_step % eval_freq == 0:
                res = evaluate_dpo_loss_loader(
                    policy_model, 
                    reference_model,
                    train_loader,
                    val_loader,
                    beta,
                    eval_iter
                )

                tracking['train_loss'].append(res['train_loss'])
                tracking['train_chosen_reward'].append(res['train_chosen_reward'])
                tracking['train_rejected_reward'].append(res['train_rejected_reward'])
                tracking['val_loss'].append(res['val_loss'])
                tracking['val_chosen_reward'].append(res['val_chosen_reward'])
                tracking['val_rejected_reward'].append(res['val_rejected_reward'])
                tracking['tokens_seen'].append(tokens_seen)
                train_reward_margin = res['train_chosen_reward'] - res['train_rejected_reward']
                val_reward_margin = res['val_chosen_reward'] - res['val_rejected_reward']

                print(
                    f"Epoch: {epoch} (Step {global_step:06d}): "
                    f"train loss: {res['train_loss']:.3f}, val loss: {res['val_loss']:.3f}"
                    f"train reward margins {train_reward_margin}"
                    f"val reward margins {val_reward_margin}"
                )

        generate_and_print_sample(policy_model, tokenizer, device, start_context)
    return tracking


In [ ]:
# checking the dpo_loss_loader
torch.manual_seed(123)

res = evaluate_dpo_loss_loader(
    policy_model, 
    reference_model,
    train_loader,
    val_loader,
    beta=0.1,
    eval_iter=5
)

print("train loss", res['train_loss'])
print("eval loss", res['val_loss'])
print(f"train_reward_margin {res['train_chosen_reward'] - res['train_rejected_reward']}")
print(f"val_reward_margin  {res['val_chosen_reward'] - res['val_rejected_reward']}")



train loss 0.6931471824645996
eval loss 0.6931471824645996
train reward margin
train_reward_margin 0.0
val_reward_margin  0.0


In [ ]:
import time

start_time = time.time()
optimizer = torch.optim.AdamW(policy_model.parameters(), lr=5e-6, weight_decay=0.01)
num_epochs = 1
tracking = train_model_dpo_simple(
    policy_model=policy_model,
    reference_model=reference_model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    num_epochs=num_epochs,
    beta=0.1,
    eval_freq=5,
    eval_iter=5,
    start_context=format_input(val_data1[2]),
    tokenizer=tokenizer
)

end_time = time.time()
print(f"Training completed in {(end_time - start_time)/60:.2f} minutes.")